# 05 — Realism, No-Copying, and Diversity Evaluation


core metrics:

```text
Realism:
- absolute mean difference
- absolute std difference
- histogram overlap
- FFT log-magnitude MAE

No-copying:
- copy ratio
- near-duplicate rate

Diversity:
- synthetic diversity ratio
```
this version supports two KoVAE synthetic methods:

```text
rollout_v1
posterior_bank_v2
```

can evaluate both methods or only one method using:

```python
EVAL_CONFIG["methods_to_evaluate"] = ["rollout_v1", "posterior_bank_v2"]
EVAL_CONFIG["methods_to_evaluate"] = ["posterior_bank_v2"]
```

The synthetic base directory is configurable:

```python
EVAL_CONFIG["synthetic_base_dir"] = "data/synthetic_subjects/kovae"
```


In [1]:

# ============================================================
# 05_realism_diversity_kovae_dual_methods.py
#
# Adapted from friend's Realism_Diversity.ipynb.
#
# Native-rate realism / no-copying / diversity evaluation.
#
# Supports:
#   - rollout_v1
#   - posterior_bank_v2
#
# It keeps the same main tables:
#
# Realism:
#   - mean difference
#   - std difference
#   - histogram overlap
#   - FFT log-magnitude MAE
#
# No copying:
#   - copy ratio
#   - near-duplicate rate
#
# Diversity:
#   - synthetic diversity ratio
#
# EDA and TEMP are evaluated separately.
# ============================================================

from __future__ import annotations

from pathlib import Path
from typing import Dict, List, Tuple

import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd


# ============================================================
# Config
# ============================================================

EVAL_CONFIG = {
    "project_root": "/home/iailab42/khans1/projects/ir",

    "real_dir": "data/processed/native_rates",
    "synthetic_base_dir": "data/synthetic_subjects/kovae",

    # Run both methods, or use only one:
    # ["rollout_v1"]
    # ["posterior_bank_v2"]
    # ["rollout_v1", "posterior_bank_v2"]
    "methods_to_evaluate": ["rollout_v1", "posterior_bank_v2"],

    "results_base_dir": "results/realism_diversity",
    "figures_base_dir": "figures/realism_diversity",

    "random_seed": 42,

    # For realism, use train by default because synthetic data was generated
    # from a model trained on train subjects.
    # Options: "train", "test", "all"
    "realism_reference_split": "train",

    "histogram_bins": 80,

    # Friend's A6000 defaults.
    # Use 1000 for final. Use 500 if laptop/runtime is slow.
    "max_real_train_windows_per_activity": 1000,
    "max_syn_windows_per_activity": 1000,
    "distance_batch_size": 256,

    # Optional summary plots in addition to friend's CSV tables.
    "save_summary_plots": True,
}


# ============================================================
# Native array configs
# ============================================================

ARRAY_CONFIGS = {
    "ACC": {
        "real_filename": "all_X_acc_32hz.npy",
        "syn_filename": "generated_subjects_X_acc_32hz.npy",
        "hz": 32,
        "window_len": 256,
        "num_channels": 3,
        "channel_names": ["ACC_x", "ACC_y", "ACC_z"],
    },
    "BVP": {
        "real_filename": "all_X_bvp_64hz.npy",
        "syn_filename": "generated_subjects_X_bvp_64hz.npy",
        "hz": 64,
        "window_len": 512,
        "num_channels": 1,
        "channel_names": ["BVP"],
    },
    "SLOW": {
        "real_filename": "all_X_slow_4hz.npy",
        "syn_filename": "generated_subjects_X_slow_4hz.npy",
        "hz": 4,
        "window_len": 32,
        "num_channels": 2,
        "channel_names": ["EDA", "TEMP"],
    },
}


# ============================================================
# Per-signal configs
# EDA and TEMP are separated here.
# ============================================================

SIGNAL_CONFIGS = {
    "ACC_x": {
        "array_key": "ACC",
        "channel_idx": 0,
        "hz": 32,
        "window_len": 256,
    },
    "ACC_y": {
        "array_key": "ACC",
        "channel_idx": 1,
        "hz": 32,
        "window_len": 256,
    },
    "ACC_z": {
        "array_key": "ACC",
        "channel_idx": 2,
        "hz": 32,
        "window_len": 256,
    },
    "BVP": {
        "array_key": "BVP",
        "channel_idx": 0,
        "hz": 64,
        "window_len": 512,
    },
    "EDA": {
        "array_key": "SLOW",
        "channel_idx": 0,
        "hz": 4,
        "window_len": 32,
    },
    "TEMP": {
        "array_key": "SLOW",
        "channel_idx": 1,
        "hz": 4,
        "window_len": 32,
    },
}

SIGNAL_NAMES = ["ACC_x", "ACC_y", "ACC_z", "BVP", "EDA", "TEMP"]
ACTIVITY_IDS = [1, 2, 3, 4, 5, 6, 7, 8]


# ============================================================
# Fixed split
# ============================================================

TRAIN_SUBJECTS = [
    "S1", "S2", "S3", "S4", "S5", "S6", "S9", "S11", "S12", "S13"
]

VAL_SUBJECTS = ["S14", "S15"]
TEST_SUBJECTS = ["S7", "S8", "S10"]


# ============================================================
# Path helpers
# ============================================================

def get_base_paths(config: Dict) -> Dict[str, Path]:
    root = Path(config["project_root"])
    return {
        "root": root,
        "real_dir": root / config["real_dir"],
        "synthetic_base_dir": root / config["synthetic_base_dir"],
        "results_base_dir": root / config["results_base_dir"],
        "figures_base_dir": root / config["figures_base_dir"],
        "configs_dir": root / "configs",
    }


def get_method_paths(base_paths: Dict[str, Path], method_name: str) -> Dict[str, Path]:
    return {
        "synthetic_dir": base_paths["synthetic_base_dir"] / method_name,
        "results_dir": base_paths["results_base_dir"] / method_name,
        "csv_dir": base_paths["results_base_dir"] / method_name / "csv",
        "figures_dir": base_paths["figures_base_dir"] / method_name,
    }


def make_dirs(*dirs: Path) -> None:
    for directory in dirs:
        directory.mkdir(parents=True, exist_ok=True)


def save_json(data: Dict, path: Path) -> None:
    path.write_text(json.dumps(data, indent=2), encoding="utf-8")


def require_file(path: Path) -> Path:
    path = Path(path)

    if not path.exists():
        raise FileNotFoundError(f"Missing file: {path}")

    return path


# ============================================================
# Basic helpers
# ============================================================

def subject_sort_key(s):
    s = str(s)

    if s.startswith("S"):
        try:
            return int(s[1:])
        except Exception:
            return s

    return s


def check_array_shape(X: np.ndarray, cfg: Dict, name: str) -> None:
    if X.ndim != 3:
        raise ValueError(f"{name}: expected [N,T,C], got {X.shape}")

    if X.shape[1] != int(cfg["window_len"]):
        raise ValueError(
            f"{name}: expected window length {cfg['window_len']}, got {X.shape[1]}"
        )

    if X.shape[2] != int(cfg["num_channels"]):
        raise ValueError(
            f"{name}: expected {cfg['num_channels']} channels, got {X.shape[2]}"
        )


def safe_mean(x):
    x = np.asarray(x)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return np.nan

    return float(np.mean(x))


def safe_std(x):
    x = np.asarray(x)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return np.nan

    return float(np.std(x))


def safe_percentile(x, q):
    x = np.asarray(x)
    x = x[np.isfinite(x)]

    if len(x) == 0:
        return np.nan

    return float(np.percentile(x, q))


# ============================================================
# Loading
# ============================================================

def load_real_common(real_dir: Path) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    real_y_path = real_dir / "all_y.npy"
    real_subject_path = real_dir / "all_subject.npy"
    real_metadata_path = real_dir / "all_metadata.csv"

    require_file(real_y_path)
    require_file(real_subject_path)

    y = np.load(real_y_path).astype(np.int64)
    subjects = np.load(real_subject_path, allow_pickle=True).astype(str)

    if real_metadata_path.exists():
        metadata = pd.read_csv(real_metadata_path)
    else:
        warnings.warn(
            f"Real metadata not found at {real_metadata_path}. "
            "Creating fallback metadata."
        )
        metadata = pd.DataFrame(index=np.arange(len(y)))

    if len(y) != len(subjects):
        raise ValueError(f"Real y/subject mismatch: {len(y)} vs {len(subjects)}")

    if len(metadata) != len(y):
        raise ValueError(f"Real metadata/y mismatch: {len(metadata)} vs {len(y)}")

    metadata = metadata.copy()
    metadata["subject"] = subjects
    metadata["activity_label"] = y
    metadata["array_index"] = np.arange(len(y), dtype=np.int64)

    return y, subjects, metadata


def load_synthetic_common(synthetic_dir: Path) -> Tuple[np.ndarray, np.ndarray, pd.DataFrame]:
    syn_y_path = synthetic_dir / "generated_subjects_all_y.npy"
    syn_subject_path = synthetic_dir / "generated_subjects_all_subject.npy"
    syn_metadata_path = synthetic_dir / "generated_subjects_metadata.csv"

    require_file(syn_y_path)
    require_file(syn_subject_path)
    require_file(syn_metadata_path)

    y = np.load(syn_y_path).astype(np.int64)
    subjects = np.load(syn_subject_path, allow_pickle=True).astype(str)
    metadata = pd.read_csv(syn_metadata_path)

    if len(y) != len(subjects):
        raise ValueError(f"Synthetic y/subject mismatch: {len(y)} vs {len(subjects)}")

    if len(metadata) != len(y):
        raise ValueError(f"Synthetic metadata/y mismatch: {len(metadata)} vs {len(y)}")

    metadata = metadata.copy()

    if "synthetic_subject" not in metadata.columns:
        metadata["synthetic_subject"] = subjects

    metadata["synthetic_subject"] = subjects
    metadata["activity_label"] = y
    metadata["array_index"] = np.arange(len(y), dtype=np.int64)

    return y, subjects, metadata


def load_array_data(real_dir: Path, synthetic_dir: Path) -> Tuple[Dict[str, np.ndarray], Dict[str, np.ndarray]]:
    real_X = {}
    syn_X = {}

    for array_key, cfg in ARRAY_CONFIGS.items():
        real_path = real_dir / cfg["real_filename"]
        syn_path = synthetic_dir / cfg["syn_filename"]

        require_file(real_path)
        require_file(syn_path)

        Xr = np.load(real_path).astype(np.float32)
        Xs = np.load(syn_path).astype(np.float32)

        check_array_shape(Xr, cfg, f"Real {array_key}")
        check_array_shape(Xs, cfg, f"Synthetic {array_key}")

        real_X[array_key] = Xr
        syn_X[array_key] = Xs

    return real_X, syn_X


def apply_mask_to_data(
    X_dict: Dict[str, np.ndarray],
    y: np.ndarray,
    subjects: np.ndarray,
    metadata: pd.DataFrame,
    mask: np.ndarray,
) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray, pd.DataFrame]:
    mask = np.asarray(mask, dtype=bool)

    X_out = {}

    for array_key, X in X_dict.items():
        X_out[array_key] = X[mask].astype(np.float32)

    y_out = y[mask].astype(np.int64)
    subjects_out = subjects[mask].astype(str)

    metadata_out = metadata.loc[mask].copy().reset_index(drop=True)
    metadata_out["array_index"] = np.arange(len(y_out), dtype=np.int64)

    return X_out, y_out, subjects_out, metadata_out


def filter_activities(
    X_dict: Dict[str, np.ndarray],
    y: np.ndarray,
    subjects: np.ndarray,
    metadata: pd.DataFrame,
) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray, pd.DataFrame]:
    keep = np.isin(y, np.array(ACTIVITY_IDS, dtype=np.int64))

    return apply_mask_to_data(
        X_dict=X_dict,
        y=y,
        subjects=subjects,
        metadata=metadata,
        mask=keep,
    )


def filter_by_subjects(
    X_dict: Dict[str, np.ndarray],
    y: np.ndarray,
    subjects: np.ndarray,
    metadata: pd.DataFrame,
    selected_subjects: List[str],
) -> Tuple[Dict[str, np.ndarray], np.ndarray, np.ndarray, pd.DataFrame]:
    selected_subjects = set(str(s) for s in selected_subjects)
    keep = np.array([str(s) in selected_subjects for s in subjects], dtype=bool)

    return apply_mask_to_data(
        X_dict=X_dict,
        y=y,
        subjects=subjects,
        metadata=metadata,
        mask=keep,
    )


def load_all_data_for_method(
    real_dir: Path,
    synthetic_dir: Path,
) -> Tuple[
    Dict[str, np.ndarray],
    np.ndarray,
    np.ndarray,
    pd.DataFrame,
    Dict[str, np.ndarray],
    np.ndarray,
    np.ndarray,
    pd.DataFrame,
]:
    print("=" * 80)
    print("Loading real native-rate data")
    print("=" * 80)

    real_y, real_subjects, real_meta = load_real_common(real_dir)

    print("=" * 80)
    print("Loading synthetic native-rate data")
    print("=" * 80)
    print("Synthetic directory:", synthetic_dir)

    syn_y, syn_subjects, syn_meta = load_synthetic_common(synthetic_dir)

    print("=" * 80)
    print("Loading native-rate arrays")
    print("=" * 80)

    real_X, syn_X = load_array_data(real_dir, synthetic_dir)

    for array_key in ARRAY_CONFIGS:
        if len(real_X[array_key]) != len(real_y):
            raise ValueError(
                f"Real {array_key}/y length mismatch: "
                f"{len(real_X[array_key])} vs {len(real_y)}"
            )

        if len(syn_X[array_key]) != len(syn_y):
            raise ValueError(
                f"Synthetic {array_key}/y length mismatch: "
                f"{len(syn_X[array_key])} vs {len(syn_y)}"
            )

        print(f"Real {array_key}:      {real_X[array_key].shape}")
        print(f"Synthetic {array_key}: {syn_X[array_key].shape}")

    real_X, real_y, real_subjects, real_meta = filter_activities(
        X_dict=real_X,
        y=real_y,
        subjects=real_subjects,
        metadata=real_meta,
    )

    syn_X, syn_y, syn_subjects, syn_meta = filter_activities(
        X_dict=syn_X,
        y=syn_y,
        subjects=syn_subjects,
        metadata=syn_meta,
    )

    return (
        real_X,
        real_y,
        real_subjects,
        real_meta,
        syn_X,
        syn_y,
        syn_subjects,
        syn_meta,
    )


# ============================================================
# Signal extraction
# ============================================================

def get_signal_windows(X_dict: Dict[str, np.ndarray], signal_name: str) -> np.ndarray:
    """
    Returns one signal as [N, T].

    ACC_x: [N, 256]
    ACC_y: [N, 256]
    ACC_z: [N, 256]
    BVP:   [N, 512]
    EDA:   [N, 32]
    TEMP:  [N, 32]
    """

    signal_cfg = SIGNAL_CONFIGS[signal_name]
    array_key = signal_cfg["array_key"]
    channel_idx = int(signal_cfg["channel_idx"])

    X = X_dict[array_key]

    return X[:, :, channel_idx].astype(np.float32)


# ============================================================
# Realism metrics
# ============================================================

def histogram_overlap(real_values: np.ndarray, syn_values: np.ndarray, bins: int) -> float:
    real_values = np.asarray(real_values, dtype=np.float32)
    syn_values = np.asarray(syn_values, dtype=np.float32)

    real_values = real_values[np.isfinite(real_values)]
    syn_values = syn_values[np.isfinite(syn_values)]

    if len(real_values) == 0 or len(syn_values) == 0:
        return np.nan

    vmin = float(min(real_values.min(), syn_values.min()))
    vmax = float(max(real_values.max(), syn_values.max()))

    if vmin == vmax:
        return 1.0

    hist_real, bin_edges = np.histogram(
        real_values,
        bins=bins,
        range=(vmin, vmax),
    )

    hist_syn, _ = np.histogram(
        syn_values,
        bins=bin_edges,
    )

    hist_real = hist_real.astype(np.float64)
    hist_syn = hist_syn.astype(np.float64)

    if hist_real.sum() == 0 or hist_syn.sum() == 0:
        return np.nan

    hist_real /= hist_real.sum()
    hist_syn /= hist_syn.sum()

    return float(np.minimum(hist_real, hist_syn).sum())


def average_log_fft(windows_1signal: np.ndarray):
    """
    windows_1signal:
        [N, T]

    Returns:
        average log-magnitude FFT [freq_bins]
    """

    windows_1signal = np.asarray(windows_1signal, dtype=np.float32)

    if len(windows_1signal) == 0:
        return None

    fft = np.fft.rfft(windows_1signal, axis=1)
    mag = np.log1p(np.abs(fft))

    return mag.mean(axis=0).astype(np.float32)


def fft_logmag_mae(real_windows_1signal: np.ndarray, syn_windows_1signal: np.ndarray) -> float:
    """
    Lower is better.
    """

    real_fft = average_log_fft(real_windows_1signal)
    syn_fft = average_log_fft(syn_windows_1signal)

    if real_fft is None or syn_fft is None:
        return np.nan

    return float(np.mean(np.abs(real_fft - syn_fft)))


def compute_realism_table(
    real_X_dict: Dict[str, np.ndarray],
    real_y: np.ndarray,
    syn_X_dict: Dict[str, np.ndarray],
    syn_y: np.ndarray,
    reference_name: str,
    method_name: str,
    config: Dict,
    csv_dir: Path,
) -> pd.DataFrame:
    """
    Keeps only relevant realism metrics:

    - real_mean
    - syn_mean
    - abs_mean_diff
    - real_std
    - syn_std
    - abs_std_diff
    - histogram_overlap
    - fft_logmag_mae
    """

    rows = []
    bins = int(config["histogram_bins"])

    for signal_name in SIGNAL_NAMES:
        signal_cfg = SIGNAL_CONFIGS[signal_name]

        real_signal = get_signal_windows(real_X_dict, signal_name)
        syn_signal = get_signal_windows(syn_X_dict, signal_name)

        print("\n" + "=" * 70)
        print(f"Realism metrics | Method: {method_name} | Signal: {signal_name}")
        print("=" * 70)

        for act in ACTIVITY_IDS:
            real_idx = np.where(real_y == int(act))[0]
            syn_idx = np.where(syn_y == int(act))[0]

            real_act = real_signal[real_idx]
            syn_act = syn_signal[syn_idx]

            print(
                f"{signal_name} | activity {act} | "
                f"real={real_act.shape} | syn={syn_act.shape}"
            )

            if len(real_act) == 0 or len(syn_act) == 0:
                continue

            real_values = real_act.reshape(-1)
            syn_values = syn_act.reshape(-1)

            real_mean = safe_mean(real_values)
            syn_mean = safe_mean(syn_values)

            real_std = safe_std(real_values)
            syn_std = safe_std(syn_values)

            row = {
                "method": method_name,
                "reference_real_split": reference_name,
                "activity": int(act),
                "signal": signal_name,
                "source_array": signal_cfg["array_key"],
                "channel_idx_in_source_array": int(signal_cfg["channel_idx"]),
                "hz": int(signal_cfg["hz"]),
                "window_len": int(signal_cfg["window_len"]),
                "real_num_windows": int(len(real_act)),
                "syn_num_windows": int(len(syn_act)),
                "real_mean": real_mean,
                "syn_mean": syn_mean,
                "abs_mean_diff_lower_is_better": abs(real_mean - syn_mean),
                "real_std": real_std,
                "syn_std": syn_std,
                "abs_std_diff_lower_is_better": abs(real_std - syn_std),
                "histogram_overlap_0_to_1_higher_is_better": histogram_overlap(
                    real_values,
                    syn_values,
                    bins=bins,
                ),
                "fft_logmag_mae_lower_is_better": fft_logmag_mae(
                    real_act,
                    syn_act,
                ),
            }

            rows.append(row)

    df = pd.DataFrame(rows)

    out_path = csv_dir / "realism_by_activity_signal.csv"
    df.to_csv(out_path, index=False)

    print("\nSaved:", out_path)

    return df


def save_realism_summary_tables(realism_df: pd.DataFrame, csv_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if realism_df is None or len(realism_df) == 0:
        print("No realism table to summarize.")
        return pd.DataFrame(), pd.DataFrame()

    metric_cols = [
        "abs_mean_diff_lower_is_better",
        "abs_std_diff_lower_is_better",
        "histogram_overlap_0_to_1_higher_is_better",
        "fft_logmag_mae_lower_is_better",
    ]

    avg_by_signal = (
        realism_df
        .groupby(["method", "signal"], as_index=False)[metric_cols]
        .mean(numeric_only=True)
    )

    counts_by_signal = (
        realism_df
        .groupby(["method", "signal"], as_index=False)[["real_num_windows", "syn_num_windows"]]
        .sum()
    )

    avg_by_signal = avg_by_signal.merge(
        counts_by_signal,
        on=["method", "signal"],
        how="left",
    )

    avg_by_signal = avg_by_signal[
        ["method", "signal", "real_num_windows", "syn_num_windows"] + metric_cols
    ]

    out_path = csv_dir / "realism_average_by_signal.csv"
    avg_by_signal.to_csv(out_path, index=False)
    print("Saved:", out_path)

    overall = realism_df.groupby("method", as_index=False)[metric_cols].mean(numeric_only=True)
    overall.insert(1, "summary", "average_across_all_activity_signal_rows")

    out_path = csv_dir / "realism_average_overall.csv"
    overall.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return avg_by_signal, overall


# ============================================================
# Nearest-neighbor helpers
# ============================================================

def sample_indices_for_activity(
    y: np.ndarray,
    activity_id: int,
    max_windows: int,
    rng: np.random.Generator,
) -> np.ndarray:
    idx = np.where(y == int(activity_id))[0]

    if len(idx) == 0:
        return idx.astype(np.int64)

    if len(idx) > max_windows:
        idx = rng.choice(idx, size=max_windows, replace=False)

    return idx.astype(np.int64)


def nearest_mse_distances(
    A: np.ndarray,
    B: np.ndarray,
    batch_size: int = 256,
    exclude_self: bool = False,
) -> np.ndarray:
    """
    For each row in A, compute nearest MSE distance to rows in B.

    A:
        [N, D]

    B:
        [M, D]

    If exclude_self=True, A and B must be the same matrix/order.
    """

    A = np.asarray(A, dtype=np.float32)
    B = np.asarray(B, dtype=np.float32)

    if A.ndim != 2 or B.ndim != 2:
        raise ValueError(f"Expected A and B to be 2D, got {A.shape}, {B.shape}")

    if A.shape[1] != B.shape[1]:
        raise ValueError(f"Feature dimension mismatch: {A.shape[1]} vs {B.shape[1]}")

    if len(A) == 0 or len(B) == 0:
        return np.array([], dtype=np.float32)

    if exclude_self:
        if len(A) != len(B):
            raise ValueError("exclude_self=True requires same number of rows.")

        if len(A) < 2:
            return np.array([], dtype=np.float32)

    D = float(A.shape[1])

    B_T = B.T
    B_norm = np.sum(B * B, axis=1)[None, :]

    nearest = []

    for start in range(0, len(A), batch_size):
        end = min(start + batch_size, len(A))
        A_batch = A[start:end]

        A_norm = np.sum(A_batch * A_batch, axis=1)[:, None]

        dist = (A_norm + B_norm - 2.0 * (A_batch @ B_T)) / D
        dist = np.maximum(dist, 0.0)

        if exclude_self:
            rows = np.arange(end - start)
            cols = np.arange(start, end)
            dist[rows, cols] = np.inf

        nearest_batch = np.min(dist, axis=1)
        nearest.append(nearest_batch.astype(np.float32))

    return np.concatenate(nearest, axis=0)


def summarize_mse(values: np.ndarray, prefix: str) -> Dict[str, float]:
    values = np.asarray(values, dtype=np.float32)

    if len(values) == 0:
        return {
            f"{prefix}_mean": np.nan,
            f"{prefix}_median": np.nan,
            f"{prefix}_p01": np.nan,
            f"{prefix}_p05": np.nan,
            f"{prefix}_p95": np.nan,
        }

    return {
        f"{prefix}_mean": safe_mean(values),
        f"{prefix}_median": safe_percentile(values, 50),
        f"{prefix}_p01": safe_percentile(values, 1),
        f"{prefix}_p05": safe_percentile(values, 5),
        f"{prefix}_p95": safe_percentile(values, 95),
    }


# ============================================================
# Copying / diversity table
# ============================================================

def compute_copy_diversity_table(
    real_train_X_dict: Dict[str, np.ndarray],
    real_train_y: np.ndarray,
    syn_X_dict: Dict[str, np.ndarray],
    syn_y: np.ndarray,
    method_name: str,
    config: Dict,
    csv_dir: Path,
) -> pd.DataFrame:
    """
    Keeps relevant copy/diversity metrics, while keeping the
    nearest-neighbor MSE values needed to calculate the ratios.

    real_train_to_nearest_real_train_mse_mean:
        Baseline natural real-real similarity.

    syn_to_nearest_train_real_mse_mean:
        Synthetic-to-training-real closeness.

    syn_to_nearest_syn_mse_mean:
        Synthetic internal diversity.

    copy_ratio_mean:
        syn_to_nearest_train_real_mse_mean /
        real_train_to_nearest_real_train_mse_mean

    near_duplicate_rate_p01_lower_is_better:
        Fraction of synthetic windows whose nearest train-real MSE is below
        the 1st percentile of real-train-to-real-train NN MSE.

    synthetic_diversity_ratio_mean:
        syn_to_nearest_syn_mse_mean /
        real_train_to_nearest_real_train_mse_mean
    """

    rng = np.random.default_rng(int(config["random_seed"]))
    rows = []

    max_real = int(config["max_real_train_windows_per_activity"])
    max_syn = int(config["max_syn_windows_per_activity"])
    batch_size = int(config["distance_batch_size"])

    for signal_name in SIGNAL_NAMES:
        signal_cfg = SIGNAL_CONFIGS[signal_name]

        real_signal = get_signal_windows(real_train_X_dict, signal_name)
        syn_signal = get_signal_windows(syn_X_dict, signal_name)

        print("\n" + "=" * 70)
        print(f"Copy/diversity metrics | Method: {method_name} | Signal: {signal_name}")
        print("=" * 70)

        for act in ACTIVITY_IDS:
            real_idx = sample_indices_for_activity(
                y=real_train_y,
                activity_id=act,
                max_windows=max_real,
                rng=rng,
            )

            syn_idx = sample_indices_for_activity(
                y=syn_y,
                activity_id=act,
                max_windows=max_syn,
                rng=rng,
            )

            real_sample = real_signal[real_idx].astype(np.float32)
            syn_sample = syn_signal[syn_idx].astype(np.float32)

            print(
                f"{signal_name} | activity {act} | "
                f"real_train_sample={real_sample.shape} | syn_sample={syn_sample.shape}"
            )

            if len(real_sample) < 2 or len(syn_sample) == 0:
                continue

            real_flat = real_sample.reshape(len(real_sample), -1).astype(np.float32)
            syn_flat = syn_sample.reshape(len(syn_sample), -1).astype(np.float32)

            real_to_real_nn = nearest_mse_distances(
                A=real_flat,
                B=real_flat,
                batch_size=batch_size,
                exclude_self=True,
            )

            syn_to_train_real_nn = nearest_mse_distances(
                A=syn_flat,
                B=real_flat,
                batch_size=batch_size,
                exclude_self=False,
            )

            syn_to_syn_nn = nearest_mse_distances(
                A=syn_flat,
                B=syn_flat,
                batch_size=batch_size,
                exclude_self=True,
            )

            real_ref_mean = safe_mean(real_to_real_nn)
            syn_train_mean = safe_mean(syn_to_train_real_nn)
            syn_syn_mean = safe_mean(syn_to_syn_nn)

            if np.isfinite(real_ref_mean) and real_ref_mean > 0:
                copy_ratio_mean = float(syn_train_mean / real_ref_mean)
                synthetic_diversity_ratio_mean = float(syn_syn_mean / real_ref_mean)
            else:
                copy_ratio_mean = np.nan
                synthetic_diversity_ratio_mean = np.nan

            real_ref_p01 = safe_percentile(real_to_real_nn, 1)

            if np.isfinite(real_ref_p01):
                near_duplicate_rate_p01 = float(
                    np.mean(syn_to_train_real_nn <= real_ref_p01)
                )
            else:
                near_duplicate_rate_p01 = np.nan

            row = {
                "method": method_name,
                "activity": int(act),
                "signal": signal_name,
                "source_array": signal_cfg["array_key"],
                "channel_idx_in_source_array": int(signal_cfg["channel_idx"]),
                "hz": int(signal_cfg["hz"]),
                "window_len": int(signal_cfg["window_len"]),
                "feature_dim": int(real_flat.shape[1]),

                "real_train_sampled_windows": int(len(real_sample)),
                "synthetic_sampled_windows": int(len(syn_sample)),

                **summarize_mse(
                    real_to_real_nn,
                    "real_train_to_nearest_real_train_mse",
                ),
                **summarize_mse(
                    syn_to_train_real_nn,
                    "syn_to_nearest_train_real_mse",
                ),
                **summarize_mse(
                    syn_to_syn_nn,
                    "syn_to_nearest_syn_mse",
                ),

                "copy_ratio_mean": copy_ratio_mean,
                "near_duplicate_threshold_real_train_p01_mse": real_ref_p01,
                "near_duplicate_rate_p01_lower_is_better": near_duplicate_rate_p01,
                "synthetic_diversity_ratio_mean": synthetic_diversity_ratio_mean,
            }

            rows.append(row)

    df = pd.DataFrame(rows)

    out_path = csv_dir / "copy_diversity_by_activity_signal.csv"
    df.to_csv(out_path, index=False)

    print("\nSaved:", out_path)

    return df


def save_copy_diversity_summary_tables(copy_df: pd.DataFrame, csv_dir: Path) -> Tuple[pd.DataFrame, pd.DataFrame]:
    if copy_df is None or len(copy_df) == 0:
        print("No copy/diversity table to summarize.")
        return pd.DataFrame(), pd.DataFrame()

    metric_cols = [
        "real_train_to_nearest_real_train_mse_mean",
        "syn_to_nearest_train_real_mse_mean",
        "syn_to_nearest_syn_mse_mean",
        "copy_ratio_mean",
        "near_duplicate_rate_p01_lower_is_better",
        "synthetic_diversity_ratio_mean",
    ]

    avg_by_signal = (
        copy_df
        .groupby(["method", "signal"], as_index=False)[metric_cols]
        .mean(numeric_only=True)
    )

    counts_by_signal = (
        copy_df
        .groupby(["method", "signal"], as_index=False)[
            ["real_train_sampled_windows", "synthetic_sampled_windows"]
        ]
        .sum()
    )

    avg_by_signal = avg_by_signal.merge(
        counts_by_signal,
        on=["method", "signal"],
        how="left",
    )

    avg_by_signal = avg_by_signal[
        ["method", "signal", "real_train_sampled_windows", "synthetic_sampled_windows"]
        + metric_cols
    ]

    out_path = csv_dir / "copy_diversity_average_by_signal.csv"
    avg_by_signal.to_csv(out_path, index=False)
    print("Saved:", out_path)

    overall = copy_df.groupby("method", as_index=False)[metric_cols].mean(numeric_only=True)
    overall.insert(1, "summary", "average_across_all_activity_signal_rows")

    out_path = csv_dir / "copy_diversity_average_overall.csv"
    overall.to_csv(out_path, index=False)
    print("Saved:", out_path)

    return avg_by_signal, overall


# ============================================================
# Split summary
# ============================================================

def save_data_split_summary(
    real_train_y: np.ndarray,
    real_train_subjects: np.ndarray,
    real_val_y: np.ndarray,
    real_val_subjects: np.ndarray,
    real_test_y: np.ndarray,
    real_test_subjects: np.ndarray,
    syn_y: np.ndarray,
    syn_subjects: np.ndarray,
    method_name: str,
    csv_dir: Path,
) -> pd.DataFrame:
    rows = []

    split_items = [
        ("real_train", real_train_y, real_train_subjects),
        ("real_val", real_val_y, real_val_subjects),
        ("real_test", real_test_y, real_test_subjects),
        ("synthetic", syn_y, syn_subjects),
    ]

    for split_name, y, subjects in split_items:
        row = {
            "method": method_name,
            "split": split_name,
            "num_windows": int(len(y)),
            "num_subjects": int(len(np.unique(subjects.astype(str)))),
            "subjects": ",".join(
                sorted(np.unique(subjects.astype(str)), key=subject_sort_key)
            ),
        }

        for act in ACTIVITY_IDS:
            row[f"activity_{act}_windows"] = int(np.sum(y == int(act)))

        rows.append(row)

    df = pd.DataFrame(rows)

    out_path = csv_dir / "data_split_summary.csv"
    df.to_csv(out_path, index=False)

    print("Saved:", out_path)

    return df


# ============================================================
# Optional summary plots
# ============================================================

def save_bar_plot(
    df: pd.DataFrame,
    x_col: str,
    y_col: str,
    title: str,
    ylabel: str,
    path: Path,
) -> None:
    if df is None or len(df) == 0:
        return

    fig, ax = plt.subplots(figsize=(10, 5))
    ax.bar(df[x_col].astype(str), df[y_col])
    ax.set_title(title)
    ax.set_xlabel(x_col)
    ax.set_ylabel(ylabel)
    ax.tick_params(axis="x", rotation=45)
    fig.tight_layout()
    fig.savefig(path, dpi=200, bbox_inches="tight")
    plt.close(fig)
    print("Saved:", path)


def save_method_summary_plots(
    realism_avg_signal: pd.DataFrame,
    copy_avg_signal: pd.DataFrame,
    figures_dir: Path,
    config: Dict,
) -> None:
    if not bool(config.get("save_summary_plots", False)):
        return

    make_dirs(figures_dir)

    if realism_avg_signal is not None and len(realism_avg_signal) > 0:
        save_bar_plot(
            df=realism_avg_signal,
            x_col="signal",
            y_col="histogram_overlap_0_to_1_higher_is_better",
            title="Histogram overlap by signal",
            ylabel="Histogram overlap",
            path=figures_dir / "realism_histogram_overlap_by_signal.png",
        )

        save_bar_plot(
            df=realism_avg_signal,
            x_col="signal",
            y_col="fft_logmag_mae_lower_is_better",
            title="FFT log-magnitude MAE by signal",
            ylabel="FFT log-magnitude MAE",
            path=figures_dir / "realism_fft_mae_by_signal.png",
        )

    if copy_avg_signal is not None and len(copy_avg_signal) > 0:
        save_bar_plot(
            df=copy_avg_signal,
            x_col="signal",
            y_col="copy_ratio_mean",
            title="Copy ratio by signal",
            ylabel="Copy ratio",
            path=figures_dir / "copy_ratio_by_signal.png",
        )

        save_bar_plot(
            df=copy_avg_signal,
            x_col="signal",
            y_col="synthetic_diversity_ratio_mean",
            title="Synthetic diversity ratio by signal",
            ylabel="Synthetic diversity ratio",
            path=figures_dir / "synthetic_diversity_ratio_by_signal.png",
        )


def save_combined_method_plots(
    combined_overall: pd.DataFrame,
    figures_base_dir: Path,
    config: Dict,
) -> None:
    if not bool(config.get("save_summary_plots", False)):
        return

    if combined_overall is None or len(combined_overall) == 0:
        return

    make_dirs(figures_base_dir)

    metric_cols = [
        "histogram_overlap_0_to_1_higher_is_better",
        "fft_logmag_mae_lower_is_better",
        "copy_ratio_mean",
        "near_duplicate_rate_p01_lower_is_better",
        "synthetic_diversity_ratio_mean",
    ]

    for metric in metric_cols:
        if metric not in combined_overall.columns:
            continue

        save_bar_plot(
            df=combined_overall,
            x_col="method",
            y_col=metric,
            title=f"Method comparison: {metric}",
            ylabel=metric,
            path=figures_base_dir / f"method_comparison_{metric}.png",
        )


# ============================================================
# One-method evaluation
# ============================================================

def evaluate_one_method(method_name: str, config: Dict) -> Dict[str, object]:
    base_paths = get_base_paths(config)
    method_paths = get_method_paths(base_paths, method_name)

    make_dirs(
        method_paths["results_dir"],
        method_paths["csv_dir"],
        method_paths["figures_dir"],
        base_paths["configs_dir"],
    )

    print("\n" + "#" * 100)
    print(f"REALISM / DIVERSITY EVALUATION METHOD: {method_name}")
    print("#" * 100)

    print("\nReal directory:", base_paths["real_dir"])
    print("Synthetic directory:", method_paths["synthetic_dir"])
    print("Output directory:", method_paths["results_dir"])
    print("CSV directory:", method_paths["csv_dir"])
    print("Figures directory:", method_paths["figures_dir"])

    print("\nSignals evaluated separately:")
    print(SIGNAL_NAMES)

    print("\nFixed real split:")
    print("Train:", TRAIN_SUBJECTS)
    print("Val:  ", VAL_SUBJECTS)
    print("Test: ", TEST_SUBJECTS)

    print("\nRealism reference split:", config["realism_reference_split"])
    print("Copying check uses real TRAIN subjects only.")

    (
        real_X_all,
        real_y_all,
        real_subjects_all,
        real_meta_all,
        syn_X_all,
        syn_y_all,
        syn_subjects_all,
        syn_meta_all,
    ) = load_all_data_for_method(
        real_dir=base_paths["real_dir"],
        synthetic_dir=method_paths["synthetic_dir"],
    )

    (
        real_train_X,
        real_train_y,
        real_train_subjects,
        real_train_meta,
    ) = filter_by_subjects(
        X_dict=real_X_all,
        y=real_y_all,
        subjects=real_subjects_all,
        metadata=real_meta_all,
        selected_subjects=TRAIN_SUBJECTS,
    )

    (
        real_val_X,
        real_val_y,
        real_val_subjects,
        real_val_meta,
    ) = filter_by_subjects(
        X_dict=real_X_all,
        y=real_y_all,
        subjects=real_subjects_all,
        metadata=real_meta_all,
        selected_subjects=VAL_SUBJECTS,
    )

    (
        real_test_X,
        real_test_y,
        real_test_subjects,
        real_test_meta,
    ) = filter_by_subjects(
        X_dict=real_X_all,
        y=real_y_all,
        subjects=real_subjects_all,
        metadata=real_meta_all,
        selected_subjects=TEST_SUBJECTS,
    )

    split_df = save_data_split_summary(
        real_train_y=real_train_y,
        real_train_subjects=real_train_subjects,
        real_val_y=real_val_y,
        real_val_subjects=real_val_subjects,
        real_test_y=real_test_y,
        real_test_subjects=real_test_subjects,
        syn_y=syn_y_all,
        syn_subjects=syn_subjects_all,
        method_name=method_name,
        csv_dir=method_paths["csv_dir"],
    )

    reference_split = config["realism_reference_split"]

    if reference_split == "train":
        realism_real_X = real_train_X
        realism_real_y = real_train_y
        realism_reference_name = "real_train"

    elif reference_split == "test":
        realism_real_X = real_test_X
        realism_real_y = real_test_y
        realism_reference_name = "real_test"

    elif reference_split == "all":
        realism_real_X = real_X_all
        realism_real_y = real_y_all
        realism_reference_name = "real_all"

    else:
        raise ValueError(
            "realism_reference_split must be one of: train, test, all"
        )

    realism_df = compute_realism_table(
        real_X_dict=realism_real_X,
        real_y=realism_real_y,
        syn_X_dict=syn_X_all,
        syn_y=syn_y_all,
        reference_name=realism_reference_name,
        method_name=method_name,
        config=config,
        csv_dir=method_paths["csv_dir"],
    )

    realism_avg_signal, realism_overall = save_realism_summary_tables(
        realism_df=realism_df,
        csv_dir=method_paths["csv_dir"],
    )

    copy_df = compute_copy_diversity_table(
        real_train_X_dict=real_train_X,
        real_train_y=real_train_y,
        syn_X_dict=syn_X_all,
        syn_y=syn_y_all,
        method_name=method_name,
        config=config,
        csv_dir=method_paths["csv_dir"],
    )

    copy_avg_signal, copy_overall = save_copy_diversity_summary_tables(
        copy_df=copy_df,
        csv_dir=method_paths["csv_dir"],
    )

    save_method_summary_plots(
        realism_avg_signal=realism_avg_signal,
        copy_avg_signal=copy_avg_signal,
        figures_dir=method_paths["figures_dir"],
        config=config,
    )

    method_summary = {
        "method": method_name,
        "synthetic_dir": str(method_paths["synthetic_dir"]),
        "results_dir": str(method_paths["results_dir"]),
        "csv_dir": str(method_paths["csv_dir"]),
        "figures_dir": str(method_paths["figures_dir"]),
        "num_synthetic_windows": int(len(syn_y_all)),
        "num_synthetic_subjects": int(len(np.unique(syn_subjects_all.astype(str)))),
        "realism_reference_split": realism_reference_name,
        "csv_outputs": {
            "realism_by_activity_signal": str(method_paths["csv_dir"] / "realism_by_activity_signal.csv"),
            "realism_average_by_signal": str(method_paths["csv_dir"] / "realism_average_by_signal.csv"),
            "realism_average_overall": str(method_paths["csv_dir"] / "realism_average_overall.csv"),
            "copy_diversity_by_activity_signal": str(method_paths["csv_dir"] / "copy_diversity_by_activity_signal.csv"),
            "copy_diversity_average_by_signal": str(method_paths["csv_dir"] / "copy_diversity_average_by_signal.csv"),
            "copy_diversity_average_overall": str(method_paths["csv_dir"] / "copy_diversity_average_overall.csv"),
            "data_split_summary": str(method_paths["csv_dir"] / "data_split_summary.csv"),
        },
    }

    save_json(method_summary, method_paths["results_dir"] / "method_evaluation_summary.json")

    print("\n" + "=" * 80)
    print(f"Finished realism/diversity evaluation for method: {method_name}")
    print("=" * 80)

    return {
        "method_summary": method_summary,
        "split_df": split_df,
        "realism_df": realism_df,
        "realism_avg_signal": realism_avg_signal,
        "realism_overall": realism_overall,
        "copy_df": copy_df,
        "copy_avg_signal": copy_avg_signal,
        "copy_overall": copy_overall,
    }


# ============================================================
# Combined summaries across methods
# ============================================================

def build_combined_outputs(
    method_outputs: Dict[str, Dict[str, object]],
    config: Dict,
) -> Dict[str, pd.DataFrame]:
    base_paths = get_base_paths(config)
    make_dirs(base_paths["results_base_dir"], base_paths["figures_base_dir"])

    realism_avg_signal_list = []
    realism_overall_list = []
    copy_avg_signal_list = []
    copy_overall_list = []
    split_list = []

    for method_name, output in method_outputs.items():
        if output["realism_avg_signal"] is not None and len(output["realism_avg_signal"]) > 0:
            realism_avg_signal_list.append(output["realism_avg_signal"])

        if output["realism_overall"] is not None and len(output["realism_overall"]) > 0:
            realism_overall_list.append(output["realism_overall"])

        if output["copy_avg_signal"] is not None and len(output["copy_avg_signal"]) > 0:
            copy_avg_signal_list.append(output["copy_avg_signal"])

        if output["copy_overall"] is not None and len(output["copy_overall"]) > 0:
            copy_overall_list.append(output["copy_overall"])

        if output["split_df"] is not None and len(output["split_df"]) > 0:
            split_list.append(output["split_df"])

    combined = {}

    if realism_avg_signal_list:
        df = pd.concat(realism_avg_signal_list, ignore_index=True)
        out_path = base_paths["results_base_dir"] / "combined_realism_average_by_signal.csv"
        df.to_csv(out_path, index=False)
        print("Saved:", out_path)
        combined["combined_realism_average_by_signal"] = df

    if realism_overall_list:
        df = pd.concat(realism_overall_list, ignore_index=True)
        out_path = base_paths["results_base_dir"] / "combined_realism_average_overall.csv"
        df.to_csv(out_path, index=False)
        print("Saved:", out_path)
        combined["combined_realism_average_overall"] = df

    if copy_avg_signal_list:
        df = pd.concat(copy_avg_signal_list, ignore_index=True)
        out_path = base_paths["results_base_dir"] / "combined_copy_diversity_average_by_signal.csv"
        df.to_csv(out_path, index=False)
        print("Saved:", out_path)
        combined["combined_copy_diversity_average_by_signal"] = df

    if copy_overall_list:
        df = pd.concat(copy_overall_list, ignore_index=True)
        out_path = base_paths["results_base_dir"] / "combined_copy_diversity_average_overall.csv"
        df.to_csv(out_path, index=False)
        print("Saved:", out_path)
        combined["combined_copy_diversity_average_overall"] = df

    if split_list:
        df = pd.concat(split_list, ignore_index=True)
        out_path = base_paths["results_base_dir"] / "combined_data_split_summary.csv"
        df.to_csv(out_path, index=False)
        print("Saved:", out_path)
        combined["combined_data_split_summary"] = df

    if "combined_realism_average_overall" in combined and "combined_copy_diversity_average_overall" in combined:
        realism = combined["combined_realism_average_overall"].copy()
        copy = combined["combined_copy_diversity_average_overall"].copy()

        realism = realism.drop(columns=["summary"], errors="ignore")
        copy = copy.drop(columns=["summary"], errors="ignore")

        method_comparison = realism.merge(copy, on="method", how="outer")

        out_path = base_paths["results_base_dir"] / "combined_method_comparison_overall.csv"
        method_comparison.to_csv(out_path, index=False)
        print("Saved:", out_path)
        combined["combined_method_comparison_overall"] = method_comparison

        save_combined_method_plots(
            combined_overall=method_comparison,
            figures_base_dir=base_paths["figures_base_dir"],
            config=config,
        )

    return combined


# ============================================================
# Main
# ============================================================

def main(config: Dict = EVAL_CONFIG) -> Dict[str, object]:
    base_paths = get_base_paths(config)

    make_dirs(
        base_paths["results_base_dir"],
        base_paths["figures_base_dir"],
        base_paths["configs_dir"],
    )

    save_json(config, base_paths["configs_dir"] / "realism_diversity_kovae_dual_methods_config.json")

    print("=" * 100)
    print("Native-rate realism / copying / diversity metrics")
    print("=" * 100)

    print("\nMethods to evaluate:", config["methods_to_evaluate"])
    print("Real directory:", base_paths["real_dir"])
    print("Synthetic base directory:", base_paths["synthetic_base_dir"])
    print("Output base directory:", base_paths["results_base_dir"])
    print("Figures base directory:", base_paths["figures_base_dir"])

    method_outputs = {}

    for method_name in config["methods_to_evaluate"]:
        method_outputs[method_name] = evaluate_one_method(
            method_name=method_name,
            config=config,
        )

    combined_outputs = build_combined_outputs(
        method_outputs=method_outputs,
        config=config,
    )

    combined_summary = {
        "methods_evaluated": config["methods_to_evaluate"],
        "real_dir": str(base_paths["real_dir"]),
        "synthetic_base_dir": str(base_paths["synthetic_base_dir"]),
        "results_base_dir": str(base_paths["results_base_dir"]),
        "figures_base_dir": str(base_paths["figures_base_dir"]),
        "important_combined_outputs": {
            "combined_method_comparison_overall": str(
                base_paths["results_base_dir"] / "combined_method_comparison_overall.csv"
            ),
            "combined_realism_average_by_signal": str(
                base_paths["results_base_dir"] / "combined_realism_average_by_signal.csv"
            ),
            "combined_copy_diversity_average_by_signal": str(
                base_paths["results_base_dir"] / "combined_copy_diversity_average_by_signal.csv"
            ),
        },
    }

    save_json(combined_summary, base_paths["results_base_dir"] / "combined_realism_diversity_summary.json")

    print("\n" + "#" * 100)
    print("All realism/diversity evaluations completed.")
    print("#" * 100)

    print("\nMain final CSV to inspect:")
    print(base_paths["results_base_dir"] / "combined_method_comparison_overall.csv")
    print(base_paths["results_base_dir"] / "combined_realism_average_by_signal.csv")
    print(base_paths["results_base_dir"] / "combined_copy_diversity_average_by_signal.csv")

    return {
        "combined_summary": combined_summary,
        "method_outputs": method_outputs,
        "combined_outputs": combined_outputs,
    }


if __name__ == "__main__":
    outputs = main(EVAL_CONFIG)


Native-rate realism / copying / diversity metrics

Methods to evaluate: ['rollout_v1', 'posterior_bank_v2']
Real directory: /home/iailab42/khans1/projects/ir/data/processed/native_rates
Synthetic base directory: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae
Output base directory: /home/iailab42/khans1/projects/ir/results/realism_diversity
Figures base directory: /home/iailab42/khans1/projects/ir/figures/realism_diversity

####################################################################################################
REALISM / DIVERSITY EVALUATION METHOD: rollout_v1
####################################################################################################

Real directory: /home/iailab42/khans1/projects/ir/data/processed/native_rates
Synthetic directory: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae/rollout_v1
Output directory: /home/iailab42/khans1/projects/ir/results/realism_diversity/rollout_v1
CSV directory: /home/iailab42/khans1/pro

## Run realism/diversity evaluation

To evaluate both methods:

```python
EVAL_CONFIG["methods_to_evaluate"] = ["rollout_v1", "posterior_bank_v2"]
```

To evaluate only one method:

```python
EVAL_CONFIG["methods_to_evaluate"] = ["posterior_bank_v2"]
```

If runtime is slow, reduce the nearest-neighbor sample size:

```python
EVAL_CONFIG["max_real_train_windows_per_activity"] = 500
EVAL_CONFIG["max_syn_windows_per_activity"] = 500
EVAL_CONFIG["distance_batch_size"] = 128
```

For final run on your server/A6000, keep:

```python
EVAL_CONFIG["max_real_train_windows_per_activity"] = 1000
EVAL_CONFIG["max_syn_windows_per_activity"] = 1000
EVAL_CONFIG["distance_batch_size"] = 256
```


In [2]:
EVAL_CONFIG["synthetic_base_dir"] = "data/synthetic_subjects/kovae"
EVAL_CONFIG["methods_to_evaluate"] = ["rollout_v1", "posterior_bank_v2"]

# Final-style settings. Lower these to 500 if runtime is slow.
EVAL_CONFIG["max_real_train_windows_per_activity"] = 1000
EVAL_CONFIG["max_syn_windows_per_activity"] = 1000
EVAL_CONFIG["distance_batch_size"] = 256

EVAL_CONFIG["save_summary_plots"] = True

outputs = main(EVAL_CONFIG)
outputs["combined_summary"]


Native-rate realism / copying / diversity metrics

Methods to evaluate: ['rollout_v1', 'posterior_bank_v2']
Real directory: /home/iailab42/khans1/projects/ir/data/processed/native_rates
Synthetic base directory: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae
Output base directory: /home/iailab42/khans1/projects/ir/results/realism_diversity
Figures base directory: /home/iailab42/khans1/projects/ir/figures/realism_diversity

####################################################################################################
REALISM / DIVERSITY EVALUATION METHOD: rollout_v1
####################################################################################################

Real directory: /home/iailab42/khans1/projects/ir/data/processed/native_rates
Synthetic directory: /home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae/rollout_v1
Output directory: /home/iailab42/khans1/projects/ir/results/realism_diversity/rollout_v1
CSV directory: /home/iailab42/khans1/pro

{'methods_evaluated': ['rollout_v1', 'posterior_bank_v2'],
 'real_dir': '/home/iailab42/khans1/projects/ir/data/processed/native_rates',
 'synthetic_base_dir': '/home/iailab42/khans1/projects/ir/data/synthetic_subjects/kovae',
 'results_base_dir': '/home/iailab42/khans1/projects/ir/results/realism_diversity',
 'figures_base_dir': '/home/iailab42/khans1/projects/ir/figures/realism_diversity',
 'important_combined_outputs': {'combined_method_comparison_overall': '/home/iailab42/khans1/projects/ir/results/realism_diversity/combined_method_comparison_overall.csv',
  'combined_realism_average_by_signal': '/home/iailab42/khans1/projects/ir/results/realism_diversity/combined_realism_average_by_signal.csv',
  'combined_copy_diversity_average_by_signal': '/home/iailab42/khans1/projects/ir/results/realism_diversity/combined_copy_diversity_average_by_signal.csv'}}